Datasets/DLP Project Datasets/DeepFashion/Img/img.zip
Datasets/DLP Project Datasets/DeepFashion/img_highres_seg.zip
Datasets/DLP Project Datasets/Look Into Person/TrainVal_images/TrainVal_images.zip
Datasets/DLP Project Datasets/Look Into Person/TrainVal_parsing_annotations/TrainVal_parsing_annotations.zip

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import zipfile, os

datasets = [
    ('/content/drive/MyDrive/Datasets/DLP Project Datasets/DeepFashion/Img/img.zip', '/content/deepfashion/img/'),
    ('/content/drive/MyDrive/Datasets/DLP Project Datasets/DeepFashion/img_highres_seg.zip', '/content/deepfashion/seg/'),
    ('/content/drive/MyDrive/Datasets/DLP Project Datasets/Look Into Person/TrainVal_images/TrainVal_images.zip', '/content/lip/images/'),
    ('/content/drive/MyDrive/Datasets/DLP Project Datasets/Look Into Person/TrainVal_parsing_annotations/TrainVal_parsing_annotations.zip', '/content/lip/annotations/'),
]

for zip_path, extract_to in datasets:
    if not os.path.exists(extract_to):
        os.makedirs(extract_to, exist_ok=True)
        print(f"Unzipping {zip_path}...")
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(extract_to)
        print(f"Done → {extract_to}")
    else:
        print(f"Already extracted: {extract_to}")

Unzipping /content/drive/MyDrive/Datasets/DLP Project Datasets/DeepFashion/Img/img.zip...
Done → /content/deepfashion/img/
Unzipping /content/drive/MyDrive/Datasets/DLP Project Datasets/DeepFashion/img_highres_seg.zip...
Done → /content/deepfashion/seg/
Unzipping /content/drive/MyDrive/Datasets/DLP Project Datasets/Look Into Person/TrainVal_images/TrainVal_images.zip...
Done → /content/lip/images/
Unzipping /content/drive/MyDrive/Datasets/DLP Project Datasets/Look Into Person/TrainVal_parsing_annotations/TrainVal_parsing_annotations.zip...
Done → /content/lip/annotations/


In [3]:
!pip install -q diffusers accelerate transformers lpips
!git clone https://github.com/nabirakhan/luxe /content/Luxe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 5.6 MB/s eta 0:00:00
Cloning into '/content/Luxe'...
remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (124/124), done.
remote: Total 149 (delta 51), reused 99 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (149/149), 10.69 MiB | 18.45 MiB/s, done.
Resolving deltas: 100% (51/51), done.


In [4]:
"""Fine-tune SD v1.5 inpainting UNet on DeepFashion.

Standard diffusion fine-tuning (NOT DreamBooth — DreamBooth is for 3-30 images).
Fine-tunes cross-attention and self-attention layers in the inpainting UNet
on DeepFashion images with clothing masks applied as inpainting masks.

Domain-specific fine-tuning on DeepFashion without prior preservation causes
catastrophic forgetting of non-clothing inpainting (faces, backgrounds).
This is intentional — it strengthens the white-box surrogate for clothing attack.
Grey-box transfer to vanilla SD v2/SDXL may be weaker as a result; grey-box
results reported honestly in eval.

Hyperparameters: AdamW lr=1e-5, 5 epochs, fp16 loading + fp32 attention grads.
Run on Colab T4. Unzip cell at top of session.

FIXES vs original:
- Mid-epoch resume: saves/restores step number so interrupted epochs continue
  from exact step rather than skipping to the next epoch
- epoch+1 bug fixed: checkpoint now stores epoch_complete flag so resume
  correctly distinguishes mid-epoch vs end-of-epoch checkpoints
- FutureWarnings fixed: torch.amp instead of torch.cuda.amp
- total_loss corrected on resume: skipped steps are not counted in avg_loss
"""

from google.colab import drive
import os
drive.mount('/content/drive')

import sys


def find_drive_base():
    candidates = [
        '/content/drive/MyDrive/Datasets/DLP Project Datasets',
        '/content/drive/MyDrive/DLP Project Datasets',
        '/content/drive/MyDrive/DLP Dataset',
    ]
    for c in candidates:
        if os.path.exists(c):
            print(f"Dataset base: {c}")
            return c
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        depth = root.replace('/content/drive/MyDrive', '').count(os.sep)
        if depth > 4:
            dirs.clear()
            continue
        if 'DeepFashion' in dirs:
            print(f"Dataset base found: {root}")
            return root
    raise FileNotFoundError("Could not find dataset folder. Check Drive is mounted.")


def find_repo_base():
    candidates = [
        '/content/Luxe/backend',
        '/content/luxe/backend',
        '/content/drive/MyDrive/Luxe/backend',
    ]
    for c in candidates:
        if os.path.exists(c):
            print(f"Repo base: {c}")
            return c
    raise FileNotFoundError("Luxe repo not found. Run: git clone https://github.com/nabirakhan/luxe /content/Luxe")


DRIVE_BASE = find_drive_base()
REPO_BASE  = find_repo_base()

sys.path.insert(0, REPO_BASE)

from pathlib import Path
import torch
from torch.optim import AdamW
from diffusers import StableDiffusionInpaintPipeline
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
# FIX: use torch.amp instead of torch.cuda.amp (no more FutureWarnings)
from torch.amp import autocast, GradScaler

from data.deepfashion_loader import DeepFashionDataset

if not torch.cuda.is_available():
    raise RuntimeError("No GPU — switch Colab runtime to T4")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")

DRIVE_CKPT_DIR = Path(f"{DRIVE_BASE}/checkpoints")
DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)

RESUME_PATH    = DRIVE_CKPT_DIR / "sd_inpaint_resume.pth"
PARTITION_FILE = f"{DRIVE_BASE}/DeepFashion/list_eval_partition.txt"

IMG_ROOT   = "/content/deepfashion/img/img"
SEG_ROOT   = "/content/deepfashion/seg/img_highres"
EPOCHS     = 5
LR         = 1e-5
BATCH_SIZE = 2
device     = "cuda"


def get_attention_params(unet):
    """Return only attention params, cast to fp32 for stable gradient flow."""
    params = []
    for name, p in unet.named_parameters():
        if "attn" in name:
            p.data = p.data.float()
            p.requires_grad_(True)
            params.append(p)
        else:
            p.requires_grad_(False)
    return params


def train():
    print("Loading SD v1.5 inpainting pipeline (fp16)...")
    pipe = StableDiffusionInpaintPipeline.from_pretrained(
        "runwayml/stable-diffusion-inpainting",
        torch_dtype=torch.float16,
        safety_checker=None,
        requires_safety_checker=False,
    )
    unet            = pipe.unet.to(device)
    vae             = pipe.vae.to(device)
    noise_scheduler = pipe.scheduler

    vae.eval()
    for p in vae.parameters():
        p.requires_grad_(False)

    attn_params = get_attention_params(unet)
    optimizer   = AdamW(attn_params, lr=LR)
    # FIX: torch.amp.GradScaler with device arg
    scaler      = GradScaler('cuda')

    dataset = DeepFashionDataset(IMG_ROOT, SEG_ROOT, PARTITION_FILE, split="train")
    loader  = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2,
    )

    enc      = pipe.text_encoder.to(device)
    null_ids = pipe.tokenizer(
        "", return_tensors="pt", padding="max_length",
        max_length=pipe.tokenizer.model_max_length,
    ).input_ids.to(device)
    with torch.no_grad():
        encoder_hidden = enc(null_ids).last_hidden_state

    # FIX: track epoch, step, and whether the epoch was fully completed
    start_epoch = 0
    start_step  = 0       # which step to resume from within the epoch
    best_loss   = float("inf")

    if RESUME_PATH.exists():
        ckpt        = torch.load(RESUME_PATH, map_location=device)
        epoch_done  = ckpt.get("epoch_complete", False)  # was epoch fully finished?
        start_epoch = ckpt["epoch"] + 1 if epoch_done else ckpt["epoch"]
        start_step  = 0 if epoch_done else ckpt.get("step", 0)
        unet.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scaler.load_state_dict(ckpt["scaler"])
        best_loss   = ckpt["best_loss"]
        if epoch_done:
            print(f"Resumed: epoch {ckpt['epoch']+1} was complete, starting epoch {start_epoch+1}")
        else:
            print(f"Resumed: mid-epoch {start_epoch+1} from step {start_step}, best_loss={best_loss:.4f}")

    for epoch in range(start_epoch, EPOCHS):
        unet.train()
        total_loss  = 0.0
        steps_done  = 0  # count only steps actually trained this epoch
        pbar        = tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

        for step, (imgs, masks) in enumerate(pbar):

            # FIX: skip already-completed steps when resuming mid-epoch
            if epoch == start_epoch and step < start_step:
                continue

            imgs  = imgs.to(device, dtype=torch.float16)
            masks = masks.to(device, dtype=torch.float16)

            with torch.no_grad():
                latents = vae.encode(imgs * 2 - 1).latent_dist.sample() * vae.config.scaling_factor

            noise     = torch.randn_like(latents)
            timesteps = torch.randint(
                0, noise_scheduler.config.num_train_timesteps,
                (latents.shape[0],), device=device,
            ).long()
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

            masked_imgs    = imgs * (1 - masks)
            masked_latents = vae.encode(masked_imgs * 2 - 1).latent_dist.sample() * vae.config.scaling_factor
            mask_resized   = torch.nn.functional.interpolate(
                masks, size=latents.shape[-2:], mode="nearest"
            )

            model_input = torch.cat([noisy_latents, mask_resized, masked_latents], dim=1)
            hidden      = encoder_hidden.expand(imgs.shape[0], -1, -1)

            # FIX: torch.amp.autocast with device_type arg
            with autocast('cuda'):
                pred_noise = unet(model_input, timesteps, encoder_hidden_states=hidden).sample
                loss       = torch.nn.functional.mse_loss(
                    pred_noise.float(), noise.float()
                )

            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            steps_done += 1
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

            if (step + 1) % 500 == 0:
                torch.save({
                    "epoch":          epoch,
                    "step":           step + 1,   # FIX: save current step
                    "epoch_complete": False,       # FIX: mid-epoch flag
                    "model":          unet.state_dict(),
                    "optimizer":      optimizer.state_dict(),
                    "scaler":         scaler.state_dict(),
                    "best_loss":      best_loss,
                }, RESUME_PATH)
                print(f"  Mid-epoch checkpoint saved (step {step+1})")

        # FIX: avg_loss only over steps actually trained (handles partial resume)
        avg_loss = total_loss / steps_done if steps_done > 0 else float("nan")
        print(f"Epoch {epoch+1}/{EPOCHS}  avg_loss={avg_loss:.4f}")

        if avg_loss < best_loss:
            best_loss = avg_loss

        # FIX: epoch_complete=True so next resume skips to next epoch correctly
        torch.save({
            "epoch":          epoch,
            "step":           0,
            "epoch_complete": True,               # FIX: epoch fully done
            "model":          unet.state_dict(),
            "optimizer":      optimizer.state_dict(),
            "scaler":         scaler.state_dict(),
            "best_loss":      best_loss,
        }, RESUME_PATH)

        # reset start_step after first resumed epoch so subsequent epochs run fully
        start_step = 0

        ckpt_path = DRIVE_CKPT_DIR / f"sd_inpaint_unet_epoch{epoch+1}.pth"
        torch.save(unet.state_dict(), ckpt_path)
        print(f"Checkpoint saved → {ckpt_path}")

    vae_path = DRIVE_CKPT_DIR / "sd_inpaint_vae.pth"
    torch.save(vae.state_dict(), vae_path)
    print(f"VAE saved → {vae_path}")
    print("SD inpainting fine-tuning complete.")


train()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset base: /content/drive/MyDrive/Datasets/DLP Project Datasets
Repo base: /content/Luxe/backend


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


GPU: Tesla T4
Memory: 15.6GB
Loading SD v1.5 inpainting pipeline (fp16)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


model_index.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

...

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DeepFashionDataset [train]: 25882 samples
Resumed: mid-epoch 3 from step 7000, best_loss=0.0785


Epoch 3/5:   0%|          | 0/12941 [00:00<?, ?it/s]

  Mid-epoch checkpoint saved (step 7500)
  Mid-epoch checkpoint saved (step 8000)
  Mid-epoch checkpoint saved (step 8500)
  Mid-epoch checkpoint saved (step 9000)
  Mid-epoch checkpoint saved (step 9500)


KeyboardInterrupt: 